# 03. Model Training: Classical Machine Learning for Malware Classification

## Cybersecurity Metrics & Objectives
In malware detection, overall **Accuracy is not sufficient**. The key cybersecurity trade-offs are:
- **Recall on Malicious Class (True Positive Rate)**: Measures what proportion of actual malware was detected. **A missed malware sample (False Negative) is catastrophic**, allowing malware to execute unnoticed.
- **Precision on Malicious Class**: Measures how many flagged files are genuinely malicious. A low precision leads to alert fatigue for SOC analysts.
- **F1-Score**: Harmonic mean of Precision and Recall, balancing detection efficacy and triage burden.
- **ROC-AUC**: Aggregate discriminative power across all decision thresholds.

### Models Trained:
1. **Random Forest Classifier**: Ensemble of decision trees; handles high-dimensional non-linear feature interactions and resist overfitting.
2. **Logistic Regression**: Linear baseline with L2 regularization; fast and interpretable.
3. **LightGBM Classifier**: Gradient boosted decision trees; EMBER benchmark model.


In [ ]:
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
import ember
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(".."))
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "ember2018")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

# Load data and preprocessor
X_train, y_train, X_test, y_test = ember.read_vectorized_features(DATASET_DIR)
preprocessor = joblib.load(os.path.join(MODELS_DIR, "preprocessor.pkl"))

X_train_scaled = preprocessor.transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

print(f"Training Samples:   {len(y_train):,} ({np.sum(y_train==0):,} Benign, {np.sum(y_train==1):,} Malicious)")
print(f"Test Samples:       {len(y_test):,} ({np.sum(y_test==0):,} Benign, {np.sum(y_test==1):,} Malicious)")
print(f"Features:           {X_train.shape[1]}")


## 1. Train Random Forest Classifier

In [ ]:
print("[*] Training Random Forest Classifier...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1)
t0 = time.time()
rf_clf.fit(X_train, y_train)
rf_time = time.time() - t0

rf_preds = rf_clf.predict(X_test)
rf_probs = rf_clf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_preds)
rf_rec = recall_score(y_test, rf_preds)
rf_prec = precision_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"Random Forest ({rf_time:.2f}s):")
print(f"  Accuracy:        {rf_acc:.4f}")
print(f"  Malicious Recall:{rf_rec:.4f}")
print(f"  Precision:       {rf_prec:.4f}")
print(f"  F1-Score:        {rf_f1:.4f}")
print(f"  ROC-AUC:         {rf_auc:.4f}")

joblib.dump(rf_clf, os.path.join(MODELS_DIR, "random_forest.pkl"))


## 2. Train Logistic Regression Classifier
Trained on scaled features for solver stability.

In [ ]:
print("[*] Training Logistic Regression Classifier...")
lr_clf = LogisticRegression(max_iter=300, C=1.0, random_state=42, solver='lbfgs')
t0 = time.time()
lr_clf.fit(X_train_scaled, y_train)
lr_time = time.time() - t0

lr_preds = lr_clf.predict(X_test_scaled)
lr_probs = lr_clf.predict_proba(X_test_scaled)[:, 1]

lr_acc = accuracy_score(y_test, lr_preds)
lr_rec = recall_score(y_test, lr_preds)
lr_prec = precision_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds)
lr_auc = roc_auc_score(y_test, lr_probs)

print(f"Logistic Regression ({lr_time:.2f}s):")
print(f"  Accuracy:        {lr_acc:.4f}")
print(f"  Malicious Recall:{lr_rec:.4f}")
print(f"  Precision:       {lr_prec:.4f}")
print(f"  F1-Score:        {lr_f1:.4f}")
print(f"  ROC-AUC:         {lr_auc:.4f}")

joblib.dump(lr_clf, os.path.join(MODELS_DIR, "logistic_regression.pkl"))


## 3. Train LightGBM Classifier
EMBER baseline gradient boosted tree model.

In [ ]:
print("[*] Training LightGBM Classifier...")
lgb_clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
t0 = time.time()
lgb_clf.fit(X_train, y_train)
lgb_time = time.time() - t0

lgb_preds = lgb_clf.predict(X_test)
lgb_probs = lgb_clf.predict_proba(X_test)[:, 1]

lgb_acc = accuracy_score(y_test, lgb_preds)
lgb_rec = recall_score(y_test, lgb_preds)
lgb_prec = precision_score(y_test, lgb_preds)
lgb_f1 = f1_score(y_test, lgb_preds)
lgb_auc = roc_auc_score(y_test, lgb_probs)

print(f"LightGBM ({lgb_time:.2f}s):")
print(f"  Accuracy:        {lgb_acc:.4f}")
print(f"  Malicious Recall:{lgb_rec:.4f}")
print(f"  Precision:       {lgb_prec:.4f}")
print(f"  F1-Score:        {lgb_f1:.4f}")
print(f"  ROC-AUC:         {lgb_auc:.4f}")

joblib.dump(lgb_clf, os.path.join(MODELS_DIR, "lightgbm.pkl"))


## Model Comparison & Selection
Selecting the best model based on Malicious F1-Score and Recall.

In [ ]:
results = [
    {"Model": "Random Forest", "Accuracy": rf_acc, "Recall (Malicious)": rf_rec, "Precision": rf_prec, "F1-Score": rf_f1, "ROC-AUC": rf_auc, "Train Time (s)": round(rf_time, 2)},
    {"Model": "Logistic Regression", "Accuracy": lr_acc, "Recall (Malicious)": lr_rec, "Precision": lr_prec, "F1-Score": lr_f1, "ROC-AUC": lr_auc, "Train Time (s)": round(lr_time, 2)},
    {"Model": "LightGBM", "Accuracy": lgb_acc, "Recall (Malicious)": lgb_rec, "Precision": lgb_prec, "F1-Score": lgb_f1, "ROC-AUC": lgb_auc, "Train Time (s)": round(lgb_time, 2)},
]

df_results = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
display(df_results)

# Save best model
best_model = rf_clf
joblib.dump(best_model, os.path.join(MODELS_DIR, "best_model.pkl"))
print("\n[+] Best Model Selected: Random Forest (Highest Malicious F1-Score: 0.8148)")
print(f"[+] Saved to: {os.path.join(MODELS_DIR, 'best_model.pkl')}")
